<a href="https://colab.research.google.com/github/dayanakumar-IT/R26-DS-010-Intelligent-Care-Support/blob/caregiver-deterioration-ai/R26_DS_010_Preprocessing_Audio_MIRSD_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# Step 1 — Mount Google Drive & Create Project Structure
# ============================================================

from google.colab import drive
from pathlib import Path

# Mount Drive
drive.mount('/content/drive')

# ------------------------------------------------------------
# Create Main Project Folder
# ------------------------------------------------------------

project_folder = Path('/content/drive/MyDrive/CareSense_Multimodal')

project_folder.mkdir(parents=True, exist_ok=True)

# Create subfolders
(project_folder / 'outputs').mkdir(exist_ok=True)
(project_folder / 'models').mkdir(exist_ok=True)
(project_folder / 'plots').mkdir(exist_ok=True)

print("Project folder created:")
print(project_folder)

print("\nSubfolders:")
print("outputs/")
print("models/")
print("plots/")

Mounted at /content/drive
Project folder created:
/content/drive/MyDrive/CareSense_Multimodal

Subfolders:
outputs/
models/
plots/


In [2]:
# ============================================================
# Step 2 — Install Required Libraries
# ============================================================

!pip install -q librosa==0.10.1
!pip install -q soundfile
!pip install -q kagglehub
!pip install -q xgboost

print("All dependencies installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.7/253.7 kB 6.8 MB/s eta 0:00:00
All dependencies installed successfully.


In [7]:
# ============================================================
# Step 2 — Install Hugging Face Audio Dataset Tools
# ============================================================

!pip install -q datasets soundfile librosa[audio] evaluate

print("Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
Dependencies installed.


In [8]:
# ============================================================
# Step 3 — Load and Inspect StressDetection_MIRSD
# ============================================================

from datasets import load_dataset

dataset = load_dataset("DynamicSuperb/StressDetection_MIRSD")

print("Dataset object:")
print(dataset)

print("\nAvailable splits:")
print(dataset.keys())

first_split = list(dataset.keys())[0]

print("\nFirst split used for inspection:", first_split)

print("\nColumn names:")
print(dataset[first_split].column_names)

print("\nFirst example:")
print(dataset[first_split][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/608 [00:00<?, ?B/s]

data/test-00000-of-00001-3d7e8bb9cda552b(…):   0%|          | 0.00/17.8M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset object:
DatasetDict({
    test: Dataset({
        features: ['file', 'audio', 'label', 'word', 'instruction'],
        num_rows: 200
    })
})

Available splits:
dict_keys(['test'])

First split used for inspection: test

Column names:
['file', 'audio', 'label', 'word', 'instruction']

First example:
{'file': 'yifen/undesirable.wav', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7d7cfcb4ff20>, 'label': 'three', 'word': 'undesirable', 'instruction': 'Please do stress detection in English word. The answer could be zero, one, two, three, four, or five.'}


In [9]:
# ============================================================
# Step 4 — Inspect MIRSD Label, Speaker, and Audio Structure
# ============================================================

import pandas as pd
import numpy as np
from collections import Counter

split = "test"
ds = dataset[split]

# ------------------------------------------------------------
# Basic dataset summary
# ------------------------------------------------------------

print("Number of samples:", len(ds))
print("Columns:", ds.column_names)

# ------------------------------------------------------------
# Label distribution
# ------------------------------------------------------------

labels = ds["label"]
label_counts = Counter(labels)

print("\nLabel distribution:")
for label, count in sorted(label_counts.items()):
    print(f"{label}: {count}")

# ------------------------------------------------------------
# Extract speaker/user from file path
# Example: yifen/undesirable.wav -> speaker = yifen
# ------------------------------------------------------------

files = ds["file"]
speakers = [f.split("/")[0] if "/" in f else "unknown" for f in files]

speaker_counts = Counter(speakers)

print("\nNumber of unique speakers:", len(speaker_counts))
print("\nSpeaker distribution:")
for speaker, count in speaker_counts.items():
    print(f"{speaker}: {count}")

# ------------------------------------------------------------
# Inspect first audio sample
# ------------------------------------------------------------

example = ds[0]
audio = example["audio"]

print("\nFirst example metadata:")
print("file:", example["file"])
print("word:", example["word"])
print("label:", example["label"])
print("instruction:", example["instruction"])

# Decode audio
audio_array = audio["array"]
sampling_rate = audio["sampling_rate"]
duration = len(audio_array) / sampling_rate

print("\nFirst audio details:")
print("Sampling rate:", sampling_rate)
print("Array shape:", audio_array.shape)
print("Duration seconds:", round(duration, 3))
print("Min amplitude:", np.min(audio_array))
print("Max amplitude:", np.max(audio_array))

Number of samples: 200
Columns: ['file', 'audio', 'label', 'word', 'instruction']

Label distribution:
five: 2
four: 11
one: 80
three: 29
two: 76
zero: 2

Number of unique speakers: 22

Speaker distribution:
yifen: 8
annar: 8
hanyin: 14
yuyuzen: 11
kevin: 11
davidson: 6
kenshin: 10
fdps: 5
roger: 14
zi: 14
geniusturtle: 10
litbee: 9
ariel: 6
titon: 6
ripple: 10
stacy: 9
bobon: 10
heycat: 5
ani: 10
tammy: 5
sophia: 7
abjones: 12

First example metadata:
file: yifen/undesirable.wav
word: undesirable
label: three
instruction: Please do stress detection in English word. The answer could be zero, one, two, three, four, or five.

First audio details:
Sampling rate: 16000
Array shape: (48000,)
Duration seconds: 3.0
Min amplitude: -0.42959595
Max amplitude: 0.6315613


In [10]:
# ============================================================
# Step 5 — Convert MIRSD Labels into Binary Stress Classes
# ============================================================

import pandas as pd
from collections import Counter

ds = dataset["test"]

# ------------------------------------------------------------
# Convert word labels to numeric stress intensity
# ------------------------------------------------------------

label_to_score = {
    "zero": 0,
    "one": 1,
    "two": 2,
    "three": 3,
    "four": 4,
    "five": 5
}

records = []

for i, sample in enumerate(ds):
    file_path = sample["file"]
    speaker_id = file_path.split("/")[0] if "/" in file_path else "unknown"

    stress_score = label_to_score[sample["label"]]

    # Binary mapping:
    # 0, 1, 2 = low stress
    # 3, 4, 5 = high stress
    binary_stress = 0 if stress_score <= 2 else 1

    records.append({
        "sample_id": i,
        "file": file_path,
        "speaker_id": speaker_id,
        "word": sample["word"],
        "original_label": sample["label"],
        "stress_score": stress_score,
        "binary_stress": binary_stress
    })

mirsd_metadata = pd.DataFrame(records)

print("MIRSD metadata shape:", mirsd_metadata.shape)

print("\nOriginal stress intensity distribution:")
print(mirsd_metadata["stress_score"].value_counts().sort_index())

print("\nBinary stress distribution:")
print(mirsd_metadata["binary_stress"].value_counts().sort_index())

print("\nSpeaker count:")
print(mirsd_metadata["speaker_id"].nunique())

print("\nPreview:")
display(mirsd_metadata.head())

MIRSD metadata shape: (200, 7)

Original stress intensity distribution:
stress_score
0     2
1    80
2    76
3    29
4    11
5     2
Name: count, dtype: int64

Binary stress distribution:
binary_stress
0    158
1     42
Name: count, dtype: int64

Speaker count:
22

Preview:


,sample_id,file,speaker_id,word,original_label,stress_score,binary_stress
0,0,yifen/undesirable.wav,yifen,undesirable,three,3,1
1,1,yifen/pneumonia.wav,yifen,pneumonia,two,2,0
2,2,yifen/unanimous.wav,yifen,unanimous,two,2,0
3,3,yifen/transmission.wav,yifen,transmission,two,2,0
4,4,yifen/plausible.wav,yifen,plausible,one,1,0


In [11]:
# ============================================================
# Step 6 — Extract Acoustic Features from MIRSD Audio
# ============================================================

import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm

def extract_audio_features(audio_array, sr=16000, n_mfcc=13, n_mels=64):
    features = {}

    # Ensure mono float array
    y = np.asarray(audio_array, dtype=np.float32)

    # MFCC features
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    for i in range(n_mfcc):
        features[f"mfcc_{i+1}_mean"] = np.mean(mfcc[i])
        features[f"mfcc_{i+1}_std"] = np.std(mfcc[i])

    # Delta MFCC features
    delta_mfcc = librosa.feature.delta(mfcc)
    for i in range(n_mfcc):
        features[f"delta_mfcc_{i+1}_mean"] = np.mean(delta_mfcc[i])
        features[f"delta_mfcc_{i+1}_std"] = np.std(delta_mfcc[i])

    # Mel spectrogram features
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    for i in range(n_mels):
        features[f"mel_{i+1}_mean"] = np.mean(mel_db[i])

    # RMS energy
    rms = librosa.feature.rms(y=y)
    features["rms_mean"] = np.mean(rms)
    features["rms_std"] = np.std(rms)

    # Zero crossing rate
    zcr = librosa.feature.zero_crossing_rate(y)
    features["zcr_mean"] = np.mean(zcr)
    features["zcr_std"] = np.std(zcr)

    # Spectral centroid
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    features["spectral_centroid_mean"] = np.mean(centroid)
    features["spectral_centroid_std"] = np.std(centroid)

    # Spectral rolloff
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    features["spectral_rolloff_mean"] = np.mean(rolloff)
    features["spectral_rolloff_std"] = np.std(rolloff)

    # Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    for i in range(12):
        features[f"chroma_{i+1}_mean"] = np.mean(chroma[i])
        features[f"chroma_{i+1}_std"] = np.std(chroma[i])

    # Fundamental frequency
    f0, _, _ = librosa.pyin(
        y,
        fmin=librosa.note_to_hz("C2"),
        fmax=librosa.note_to_hz("C7"),
        sr=sr
    )

    f0_clean = f0[~np.isnan(f0)] if f0 is not None else np.array([])

    features["f0_mean"] = np.mean(f0_clean) if len(f0_clean) > 0 else 0
    features["f0_std"] = np.std(f0_clean) if len(f0_clean) > 0 else 0

    return features


feature_rows = []

for i in tqdm(range(len(ds)), desc="Extracting MIRSD features"):
    sample = ds[i]
    metadata_row = mirsd_metadata.iloc[i]

    audio = sample["audio"]
    audio_array = audio["array"]
    sr = audio["sampling_rate"]

    features = extract_audio_features(audio_array, sr=sr)

    features["sample_id"] = metadata_row["sample_id"]
    features["file"] = metadata_row["file"]
    features["speaker_id"] = metadata_row["speaker_id"]
    features["word"] = metadata_row["word"]
    features["stress_score"] = metadata_row["stress_score"]
    features["binary_stress"] = metadata_row["binary_stress"]

    feature_rows.append(features)

mirsd_features_df = pd.DataFrame(feature_rows)

print("Feature dataframe shape:", mirsd_features_df.shape)
print("\nColumns:")
print(mirsd_features_df.columns.tolist())

print("\nBinary stress distribution:")
print(mirsd_features_df["binary_stress"].value_counts().sort_index())

display(mirsd_features_df.head())


Extracting MIRSD features: 100%|██████████| 200/200 [03:38<00:00,  1.09s/it]

Feature dataframe shape: (200, 156)

Columns:
['mfcc_1_mean', 'mfcc_1_std', 'mfcc_2_mean', 'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std', 'mfcc_4_mean', 'mfcc_4_std', 'mfcc_5_mean', 'mfcc_5_std', 'mfcc_6_mean', 'mfcc_6_std', 'mfcc_7_mean', 'mfcc_7_std', 'mfcc_8_mean', 'mfcc_8_std', 'mfcc_9_mean', 'mfcc_9_std', 'mfcc_10_mean', 'mfcc_10_std', 'mfcc_11_mean', 'mfcc_11_std', 'mfcc_12_mean', 'mfcc_12_std', 'mfcc_13_mean', 'mfcc_13_std', 'delta_mfcc_1_mean', 'delta_mfcc_1_std', 'delta_mfcc_2_mean', 'delta_mfcc_2_std', 'delta_mfcc_3_mean', 'delta_mfcc_3_std', 'delta_mfcc_4_mean', 'delta_mfcc_4_std', 'delta_mfcc_5_mean', 'delta_mfcc_5_std', 'delta_mfcc_6_mean', 'delta_mfcc_6_std', 'delta_mfcc_7_mean', 'delta_mfcc_7_std', 'delta_mfcc_8_mean', 'delta_mfcc_8_std', 'delta_mfcc_9_mean', 'delta_mfcc_9_std', 'delta_mfcc_10_mean', 'delta_mfcc_10_std', 'delta_mfcc_11_mean', 'delta_mfcc_11_std', 'delta_mfcc_12_mean', 'delta_mfcc_12_std', 'delta_mfcc_13_mean', 'delta_mfcc_13_std', 'mel_1_mean', 'mel_2_mean',

,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,mfcc_3_mean,mfcc_3_std,mfcc_4_mean,mfcc_4_std,mfcc_5_mean,mfcc_5_std,...,chroma_12_mean,chroma_12_std,f0_mean,f0_std,sample_id,file,speaker_id,word,stress_score,binary_stress
0,-455.422363,147.167618,72.959969,68.124138,3.238297,36.917572,19.321304,30.197264,-2.248423,30.440174,...,0.602163,0.398463,267.533403,242.249366,0,yifen/undesirable.wav,yifen,undesirable,3,1
1,-480.474304,117.220703,55.971977,59.980965,2.857965,23.303089,14.463084,24.438894,-5.113927,29.050289,...,0.692868,0.372952,282.617915,222.857205,1,yifen/pneumonia.wav,yifen,pneumonia,2,0
2,-471.351318,126.690002,47.956638,52.786922,0.783202,17.054409,20.374908,24.666758,-5.499037,23.769880,...,0.687279,0.350583,213.258022,23.868363,2,yifen/unanimous.wav,yifen,unanimous,2,0
3,-446.823425,128.690872,45.194134,46.548260,13.922610,26.748260,29.870857,34.117371,1.091772,30.690056,...,0.619895,0.387555,280.324693,391.888704,3,yifen/transmission.wav,yifen,transmission,2,0
4,-462.389343,141.681290,66.506401,70.364784,2.559263,33.483086,15.267097,25.733881,0.160646,31.336525,...,0.762497,0.309122,198.910561,196.773488,4,yifen/plausible.wav,yifen,plausible,1,0


In [12]:
# ============================================================
# Step 7 — Speaker-Independent Train/Test Split
# ============================================================

from sklearn.model_selection import GroupShuffleSplit
from collections import Counter

# ------------------------------------------------------------
# Separate Features and Labels
# ------------------------------------------------------------

metadata_cols = [
    "sample_id",
    "file",
    "speaker_id",
    "word",
    "stress_score",
    "binary_stress"
]

feature_cols = [
    col for col in mirsd_features_df.columns
    if col not in metadata_cols
]

X = mirsd_features_df[feature_cols]
y = mirsd_features_df["binary_stress"]

groups = mirsd_features_df["speaker_id"]

print("Feature matrix shape:", X.shape)
print("Labels shape:", y.shape)

# ------------------------------------------------------------
# Speaker-Independent Split
# ------------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_speakers = set(groups.iloc[train_idx])
test_speakers = set(groups.iloc[test_idx])

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("\n==============================")
print("SPEAKER-INDEPENDENT SPLIT")
print("==============================")

print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])

print("\nTraining speakers:", len(train_speakers))
print("Testing speakers :", len(test_speakers))

print("\nShared speakers:")
print(train_speakers.intersection(test_speakers))

print("\nTraining class distribution:")
print(Counter(y_train))

print("\nTesting class distribution:")
print(Counter(y_test))

Feature matrix shape: (200, 150)
Labels shape: (200,)

SPEAKER-INDEPENDENT SPLIT
Training samples: 147
Testing samples : 53

Training speakers: 17
Testing speakers : 5

Shared speakers:
set()

Training class distribution:
Counter({0: 118, 1: 29})

Testing class distribution:
Counter({0: 40, 1: 13})


In [13]:
# ============================================================
# Step 8 — Leakage-Free Scaling + PCA Processing
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pandas as pd

# ------------------------------------------------------------
# Separate Mel and Non-Mel Features
# ------------------------------------------------------------

mel_cols = [col for col in X_train.columns if col.startswith("mel_")]
non_mel_cols = [col for col in X_train.columns if not col.startswith("mel_")]

print("Mel features:", len(mel_cols))
print("Non-Mel features:", len(non_mel_cols))

# ------------------------------------------------------------
# Scale Mel Features
# Fit ONLY on training data
# ------------------------------------------------------------

mel_scaler = StandardScaler()

X_train_mel_scaled = mel_scaler.fit_transform(X_train[mel_cols])
X_test_mel_scaled = mel_scaler.transform(X_test[mel_cols])

# ------------------------------------------------------------
# PCA on Mel Features
# Fit ONLY on training data
# ------------------------------------------------------------

pca = PCA(n_components=0.95, random_state=42)

X_train_mel_pca = pca.fit_transform(X_train_mel_scaled)
X_test_mel_pca = pca.transform(X_test_mel_scaled)

print("\nPCA components selected:", pca.n_components_)

# ------------------------------------------------------------
# Scale Non-Mel Features
# ------------------------------------------------------------

non_mel_scaler = StandardScaler()

X_train_non_mel = non_mel_scaler.fit_transform(
    X_train[non_mel_cols]
)

X_test_non_mel = non_mel_scaler.transform(
    X_test[non_mel_cols]
)

# ------------------------------------------------------------
# Combine Processed Features
# ------------------------------------------------------------

X_train_processed = pd.concat([
    pd.DataFrame(X_train_non_mel).reset_index(drop=True),
    pd.DataFrame(X_train_mel_pca).reset_index(drop=True)
], axis=1)

X_test_processed = pd.concat([
    pd.DataFrame(X_test_non_mel).reset_index(drop=True),
    pd.DataFrame(X_test_mel_pca).reset_index(drop=True)
], axis=1)

print("\nProcessed training shape:", X_train_processed.shape)
print("Processed testing shape :", X_test_processed.shape)

Mel features: 64
Non-Mel features: 86

PCA components selected: 4

Processed training shape: (147, 90)
Processed testing shape : (53, 90)


In [15]:
# ============================================================
# Fix Processed Feature Matrices for Model Training
# ============================================================

import numpy as np

# Convert to clean numpy arrays
X_train_processed = np.asarray(X_train_processed)
X_test_processed = np.asarray(X_test_processed)

print("Training shape:", X_train_processed.shape)
print("Testing shape :", X_test_processed.shape)

print("\nData type:")
print(type(X_train_processed))
print(type(X_test_processed))

Training shape: (147, 90)
Testing shape : (53, 90)

Data type:
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [16]:
# ============================================================
# Step 9 — Train and Evaluate Baseline Models
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score
)

import pandas as pd

# ------------------------------------------------------------
# Calculate imbalance weight for XGBoost
# ------------------------------------------------------------

negative_class = (y_train == 0).sum()
positive_class = (y_train == 1).sum()

scale_pos_weight = negative_class / positive_class

print("scale_pos_weight:", round(scale_pos_weight, 2))

# ------------------------------------------------------------
# Define Models
# ------------------------------------------------------------

models = {
    "Logistic Regression": LogisticRegression(
        class_weight='balanced',
        max_iter=5000,
        random_state=42
    ),

    "SVM": SVC(
        class_weight='balanced',
        probability=True,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        random_state=42
    )
}

# ------------------------------------------------------------
# Train + Evaluate
# ------------------------------------------------------------

results = []

for model_name, model in models.items():

    print("\n===================================================")
    print(model_name)
    print("===================================================")

    # Train
    model.fit(X_train_processed, y_train)

    # Predict
    y_pred = model.predict(X_test_processed)

    # Probability scores
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test_processed)[:, 1]
    else:
        y_prob = None

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    if y_prob is not None:
        roc_auc = roc_auc_score(y_test, y_prob)
    else:
        roc_auc = None

    print(f"Accuracy          : {accuracy:.4f}")
    print(f"Balanced Accuracy : {balanced_acc:.4f}")
    print(f"F1 Score          : {f1:.4f}")

    if roc_auc is not None:
        print(f"ROC-AUC           : {roc_auc:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Balanced Accuracy": balanced_acc,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    })

# ------------------------------------------------------------
# Final Results Table
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

print("\n===================================================")
print("FINAL MODEL COMPARISON")
print("===================================================")

display(results_df.sort_values(
    by="F1 Score",
    ascending=False
))

scale_pos_weight: 4.07

Logistic Regression
Accuracy          : 0.6415
Balanced Accuracy : 0.6587
F1 Score          : 0.4865
ROC-AUC           : 0.6769

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.62      0.72        40
           1       0.38      0.69      0.49        13

    accuracy                           0.64        53
   macro avg       0.62      0.66      0.61        53
weighted avg       0.74      0.64      0.67        53

Confusion Matrix:
[[25 15]
 [ 4  9]]

SVM
Accuracy          : 0.7170
Balanced Accuracy : 0.6048
F1 Score          : 0.4000
ROC-AUC           : 0.6962

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.82      0.81        40
           1       0.42      0.38      0.40        13

    accuracy                           0.72        53
   macro avg       0.61      0.60      0.61        53
weighted avg       0.71      0.72      0.71        

,Model,Accuracy,Balanced Accuracy,F1 Score,ROC-AUC
0,Logistic Regression,0.641509,0.658654,0.486486,0.676923
3,XGBoost,0.716981,0.656731,0.482759,0.707692
1,SVM,0.716981,0.604808,0.400000,0.696154
2,Random Forest,0.754717,0.525962,0.133333,0.699038
